In [1]:
!pip install implicit scipy

import pandas as pd
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
import warnings
warnings.filterwarnings("ignore")

A:\ANACONDA\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load ratings
ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    encoding="latin-1"
)

# Load movies list
movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1"
)

In [3]:
# Convert explicit rating → implicit confidence
ratings["confidence"] = 1 + (ratings["rating"] / 5) * 40

In [4]:
# USERS
user_ids = ratings["user_id"].unique()
user_to_index = {u: i for i, u in enumerate(user_ids)}
index_to_user = {i: u for u, i in user_to_index.items()}

# MOVIES (IMPORTANT — SORT BY movie_id)
movie_ids = sorted(ratings["movie_id"].unique())

movie_to_index = {movie_ids[i]: i for i in range(len(movie_ids))}
index_to_movie = {i: movie_ids[i] for i in range(len(movie_ids))}

In [5]:
rows = ratings["user_id"].map(user_to_index).astype(int)
cols = ratings["movie_id"].map(movie_to_index).astype(int)
data = ratings["confidence"].astype(float)

interaction_matrix = sp.csr_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids), len(movie_ids)),
    dtype=float
)

In [6]:
model = AlternatingLeastSquares(
    factors=50,
    regularization=0.1,
    iterations=20,
    random_state=42
)

# ALS requires item-user matrix
item_user_matrix = interaction_matrix.T.tocsr()

model.fit(item_user_matrix)

100%|██████████| 20/20 [00:14<00:00,  1.34it/s]


In [7]:
def recommend_for_user(user_id, n=10):

    if user_id not in user_to_index:
        raise ValueError("Invalid user ID")

    user_idx = user_to_index[user_id]
    user_interactions = interaction_matrix[user_idx].tocsr()

    valid_results = []
    seen_movies = set()

    request_N = n
    max_rounds = 10
    rounds = 0

    while len(valid_results) < n and rounds < max_rounds:
        rounds += 1

        movie_indices, scores = model.recommend(
            userid=user_idx,
            user_items=user_interactions,
            N=request_N,
            filter_already_liked_items=True
        )

        for internal_idx, score in zip(movie_indices, scores):

            # Map ALS internal index → movie_id
            if internal_idx >= len(index_to_movie):
                continue

            movie_id = index_to_movie[int(internal_idx)]

            if movie_id in seen_movies:
                continue
            seen_movies.add(movie_id)

            row = movies[movies["movie_id"] == movie_id]
            if row.empty:
                continue

            title = row["title"].values[0]
            valid_results.append((title, float(score)))

            if len(valid_results) == n:
                break

        request_N *= 2

    return valid_results[:n]

In [8]:
user_id = 1
top10 = recommend_for_user(user_id, n=10)

print("\nTop 10 Recommendations:\n")
for t, s in top10:
    print(f"{t}  ->  {round(s, 3)}")


Top 10 Recommendations:

Road Trip (2000)  ->  1.202
Bottle Rocket (1996)  ->  1.181
Don't Look in the Basement! (1973)  ->  1.177
Papillon (1973)  ->  1.172
Rules of Engagement (2000)  ->  1.167
Trial by Jury (1994)  ->  1.164
Problem Child 2 (1991)  ->  1.163
Wend Kuuni (God's Gift) (1982)  ->  1.153
It Came from Beneath the Sea (1955)  ->  1.148
Coming Apart (1969)  ->  1.14


In [9]:
user_id = 80
top10 = recommend_for_user(user_id, n=10)

print("\nTop 10 Recommendations:\n")
for t, s in top10:
    print(f"{t}  ->  {round(s, 3)}")


Top 10 Recommendations:

Champ, The (1979)  ->  1.257
Sirens (1994)  ->  1.166
Wolf Man, The (1941)  ->  1.126
War, The (1994)  ->  1.106
Retro Puppetmaster (1999)  ->  1.101
Bells, The (1926)  ->  1.095
It Could Happen to You (1994)  ->  1.076
Original Kings of Comedy, The (2000)  ->  1.073
Return of Martin Guerre, The (Retour de Martin Guerre, Le) (1982)  ->  1.064
It's My Party (1995)  ->  1.063


In [33]:
#EVALUATION

In [27]:
def get_recommended_items(user_idx, k):
    recs, _ = model.recommend(
        userid=user_idx,
        user_items=train_matrix[user_idx],
        N=k,
        filter_already_liked_items=True
    )
    return list(recs)   # convert numpy array → list

In [28]:
def precision_at_k(recommended, test_item, k):
    return 1.0 if test_item in recommended[:k] else 0.0

def recall_at_k(recommended, test_item, k):
    return 1.0 if test_item in recommended[:k] else 0.0

In [34]:
def apk(recommended, test_item, k):
    if test_item in recommended[:k]:
        rank = recommended[:k].index(test_item)  
        return 1.0 / (rank + 1)
    return 0.0

In [35]:
K = 10
precisions = []
recalls = []
apks = []

for user_idx, test_item in test_items.items():     # user_idx is ALS index (0–3705)

    if user_idx >= model.user_factors.shape[0]:    # safety check
        continue

    recommended = get_recommended_items(user_idx, K)

    precisions.append(precision_at_k(recommended, test_item, K))
    recalls.append(recall_at_k(recommended, test_item, K))
    apks.append(apk(recommended, test_item, K))

In [36]:
print("Precision@10:", np.mean(precisions))
print("Recall@10:", np.mean(recalls))
print("MAP@10:", np.mean(apks))

Precision@10: 0.0016189962223421479
Recall@10: 0.0016189962223421479
MAP@10: 0.0002502377097628042
